In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
from io import StringIO

from astropy.time import Time
from datetime import datetime, timedelta, date

import matplotlib.pyplot as plt
from matplotlib import ticker
from matplotlib.dates import DateFormatter
from matplotlib.ticker import MaxNLocator
import matplotlib.cm as cm
from matplotlib.colors import Normalize

from lsst_efd_client import EfdClient
from lsst.summit.utils.efdUtils import getEfdData
from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState, TMAEvent

In [ ]:
info(EfdClient)

In [ ]:
EFD_client = EfdClient("usdf_efd")

In [ ]:
# t_start = Time("2025-10-20 22:00:00")
# t_end = Time("2025-10-21 12:00:00")
t_start = Time("2025-10-22 22:00:00")
t_end = Time("2025-10-23 12:00:00")

df_dimm_meas = getEfdData(
    EFD_client, "lsst.sal.DIMM.logevent_dimmMeasurement", begin=t_start, end=t_end
)
df_dimm_meas.dropna(inplace=True)

df_ess_airflow = getEfdData(
    EFD_client, "lsst.sal.ESS.airFlow", begin=t_start, end=t_end
)

In [ ]:
df_dimm_meas = df_dimm_meas[df_dimm_meas["fwhm"] < 5]

In [ ]:
cut_1 = df_dimm_meas["salIndex"] == 1
cut_2 = df_dimm_meas["salIndex"] == 2
fig, ax = plt.subplots(1, 1, dpi=128, figsize=(14, 6))
ax.scatter(
    df_dimm_meas.index[cut_1],
    df_dimm_meas["fwhm"][cut_1],
    marker="o",
    s=3,
    color="red",
    label="DIMM 1",
)
ax.scatter(
    df_dimm_meas.index[cut_2],
    df_dimm_meas["fwhm"][cut_2],
    marker="o",
    s=3,
    color="blue",
    label="DIMM 2",
)
ax.set_xlabel("Time")
ax.set_ylabel("FWHM (arcsec)")

plt.legend()

In [ ]:
# Determine the time window where we get DIMM measuremnent
start_dimm = df_dimm_meas.index[0]
end_dimm = df_dimm_meas.index[-1]

In [ ]:
airflow = df_ess_airflow[start_dimm:end_dimm][df_ess_airflow["salIndex"] == 301]

In [ ]:
wind_speed = airflow["speed"].resample("1min").mean()
# wind_direction = airflow["direction"].resample('1min').mean()  ## Wrong due to 0-360 degrees boundary
dimm_1 = df_dimm_meas["fwhm"][cut_1].resample("1min").mean()
dimm_2 = df_dimm_meas["fwhm"][cut_2].resample("1min").mean()

In [ ]:
# Correctly handle the 0 - 360 degrees transition

# Compute sin and cos components
sin_mean = (
    airflow["direction"].apply(lambda x: np.sin(np.radians(x))).resample("1min").mean()
)
cos_mean = (
    airflow["direction"].apply(lambda x: np.cos(np.radians(x))).resample("1min").mean()
)

# Compute mean direction (in degrees)
wind_direction = np.degrees(np.arctan2(sin_mean, cos_mean))

# Ensure range is 0–360
wind_direction = (wind_direction + 360) % 360

In [ ]:
t_min = max(
    [wind_speed.index[0], wind_direction.index[0], dimm_1.index[0], dimm_2.index[0]]
)
t_max = min(
    [wind_speed.index[-1], wind_direction.index[-1], dimm_1.index[-1], dimm_2.index[-1]]
)
print(t_min, t_max)

In [ ]:
cut_1 = df_dimm_meas["salIndex"] == 1
cut_2 = df_dimm_meas["salIndex"] == 2
fig, ax = plt.subplots(2, 1, dpi=128, figsize=(8, 6))

ax[0].scatter(
    df_dimm_meas.index[cut_1],
    df_dimm_meas["fwhm"][cut_1],
    marker="o",
    s=2,
    color="red",
    label="DIMM 1",
)
ax[0].scatter(
    df_dimm_meas.index[cut_2],
    df_dimm_meas["fwhm"][cut_2],
    marker="o",
    s=2,
    color="blue",
    label="DIMM 2",
)
ax[0].set_xlabel("Time")
ax[0].set_ylabel("FWHM (arcsec)")
ax[0].legend()

Q0 = ax[1].quiver(
    wind_speed.index,
    wind_speed.values,
    wind_speed.values,
    wind_speed.values,
    angles=270.0 - wind_direction.values,
    scale=15,
    scale_units="y",
    width=0.0005,
    color="green",
)
qk0 = ax[1].quiverkey(
    Q0, 0.1, 0.9, 5, "5 m/s", labelpos="E", coordinates="axes", color="green"
)
ax[1].set_xlabel("Time")
ax[1].set_ylabel("Wind Speed (m/s)")

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 1, dpi=128, figsize=(10, 6))
Q0 = ax.quiver(
    wind_speed[t_min:t_max].values,
    dimm_1[t_min:t_max].values - dimm_2[t_min:t_max].values,
    wind_speed[t_min:t_max].values,
    wind_speed[t_min:t_max].values,
    angles=270.0 - wind_direction[t_min:t_max].values,
    scale=15,
    scale_units="x",
    width=0.001,
    color="blue",
)
ax.set_ylim([-0.5, 1.2])
ax.set_xlabel("Wind speed (m/s)")
ax.set_ylabel("DIMM-1 - DIMM-2 (arcsec)")

In [ ]:
# Try plotting in polar coordinates

wind_direction_rad = np.deg2rad(wind_direction[t_min:t_max].values)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, polar=True)

# Compass-like orientation
ax.set_theta_zero_location("N")  # 0° at North
ax.set_theta_direction(-1)  # Clockwise

# Set grid labels in degrees instead of radians
ax.set_thetagrids(range(0, 360, 30))  # every 30°


# Plot histogram bars on polar axis
sc = ax.scatter(
    wind_direction_rad,
    dimm_1[t_min:t_max].values - dimm_2[t_min:t_max].values,
    c=wind_speed[t_min:t_max].values,
    cmap="viridis",
    s=30,
)
ax.set_rlabel_position(90)

# Add a colorbar
cbar = plt.colorbar(sc, ax=ax, pad=0.1)
cbar.set_label("Wind speed (m/s)")

# Add date
date_start = t_min.strftime("%Y-%m-%d")
date_end = t_max.strftime("%Y-%m-%d")

fig.suptitle(f"FWHM(DIMM_1)- FWHM(DIMM_2) (arcsec)  / {date_start} to {date_end}")
fig.tight_layout()
plt.savefig(f"dimm_diff_{date_start}-{date_end}.png")

In [ ]:
df_dimm_meas.keys()

In [ ]:
df_dimm_meas

In [ ]:
df_dimm_gotoRaDec = getEfdData(
    EFD_client, "lsst.sal.DIMM_status", begin=t_start, end=t_end
)